# Hierarchical Relaxation Labeling (HRL) with Cantor-Set Adaptive Subdivision
### Geometric Primitive Extraction (Man-Made vs. Organic Noise)

This notebook implements the theoretical framework conceived at Grady Hospital: combining **Hierarchical Relaxation Labeling (HRL)** with **Cantor-set middle-thirds recursive label expansion** to continuously fit man-made geometric primitives (ellipses) while suppressing organic, fractal background clutter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Set random seed for reproducibility
np.random.seed(42)

## 1. Synthetic Scene Generation: Man-Made Ellipse vs. Organic Noise
We construct a 2D point cloud containing a structured man-made ellipse (low entropy, clean geometric constraints) embedded in heavy fractal/organic noise (high entropy, random scatter).

In [ ]:
def generate_synthetic_scene(n_noise=150, ellipse_params=(50, 50, 30, 15, np.pi/6), noise_bounds=(0, 100)):
    xc, yc, a, b, theta = ellipse_params
    
    # Generate clean points along the ellipse perimeter
    angles = np.linspace(0, 2 * np.pi, 60)
    ex = xc + a * np.cos(angles) * np.cos(theta) - b * np.sin(angles) * np.sin(theta)
    ey = yc + a * np.cos(angles) * np.sin(theta) + b * np.sin(angles) * np.cos(theta)
    
    # Add slight sensor noise to man-made points
    ex += np.random.normal(0, 1.0, size=ex.shape)
    ey += np.random.normal(0, 1.0, size=ey.shape)
    man_made_points = np.column_stack((ex, ey))
    
    # Generate organic / fractal noise points (scattered randomly across the canvas)
    nx = np.random.uniform(noise_bounds[0], noise_bounds[1], n_noise)
    ny = np.random.uniform(noise_bounds[0], noise_bounds[1], n_noise)
    organic_points = np.column_stack((nx, ny))
    
    # Combine scene
    scene_points = np.vstack((man_made_points, organic_points))
    
    return scene_points, man_made_points, organic_points

scene_points, true_ellipse_pts, organic_pts = generate_synthetic_scene()

plt.figure(figsize=(8, 8))
plt.scatter(organic_pts[:, 0], organic_pts[:, 1], c='forestgreen', alpha=0.5, label='Organic Noise (Nature)')
plt.scatter(true_ellipse_pts[:, 0], true_ellipse_pts[:, 1], c='dodgerblue', s=30, label='Man-Made Ellipse Perimeter')
plt.title('Synthetic Scene: Organic Clutter vs. Man-Made Geometry')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.show()

## 2. HRL Compatibility & Cantor-Set Adaptive Subdivision Engine
Instead of a massive uniform grid of candidate ellipses, we start with a coarse discrete label set of parameter estimates $(x_c, y_c, a, b)$. As probabilities concentrate around high-consistency regions, we spawn new intermediate labels using **Cantor middle-third interval splitting** to achieve continuous precision.

In [ ]:
class CantorHRLConstraintEngine:
    def __init__(self, points):
        self.points = points
        
    def evaluate_compatibility(self, label_params):
        """Compute local-global consistency score for an ellipse hypothesis."""
        xc, yc, a, b = label_params
        if a <= 2 or b <= 2 or a > 80 or b > 80:
            return 0.0
            
        # Algebraic distance metric for points relative to ellipse equation
        # (x-xc)^2 / a^2 + (y-yc)^2 / b^2 = 1
        vals = ((self.points[:, 0] - xc)**2) / (a**2) + ((self.points[:, 1] - yc)**2) / (b**2)
        residuals = np.abs(vals - 1.0)
        
        # Count points lying close to the boundary (supporting the hypothesis)
        support_count = np.sum(residuals < 0.2)
        
        # Reward tight residual clustering (man-made geometric precision)
        mean_residual = np.mean(np.minimum(residuals, 1.0))
        score = support_count * np.exp(-2.0 * mean_residual)
        return float(score)

    def cantor_middle_third_split(self, param_bounds):
        """Generate refined candidate labels via Cantor base-3 middle-third subdivision."""
        new_bounds = []
        for low, high in param_bounds:
            length = high - low
            # Cantor middle third removal leaves two intervals: [low, low + 1/3*len] and [high - 1/3*len, high]
            # We sample sub-intervals or midpoints for continuous refinement
            mid1 = low + length / 3.0
            mid2 = high - length / 3.0
            new_bounds.append((low, mid1))
            new_bounds.append((mid1, mid2))
            new_bounds.append((mid2, high))
        return new_bounds

## 3. Running the Adaptive Hierarchical Relaxation Loop

In [ ]:
engine = CantorHRLConstraintEngine(scene_points)

# Coarse initial search bounds for [xc, yc, a, b]
current_bounds = [
    (30.0, 70.0), # xc
    (30.0, 70.0), # yc
    (10.0, 50.0), # a
    (5.0,  30.0)  # b
]

iterations = 4
best_hypothesis = None
best_score = -1

print("Starting Cantor-HRL Multi-Scale Optimization...")
for it in range(iterations):
    # Sample center points of current bounds
    candidate_labels = [
        ((b[0]+b[1])/2.0) for b in current_bounds
    ]
    # To explore properly, generate grid combinations across bounds
    xc_vals = np.linspace(current_bounds[0][0], current_bounds[0][1], 5)
    yc_vals = np.linspace(current_bounds[1][0], current_bounds[1][1], 5)
    a_vals  = np.linspace(current_bounds[2][0], current_bounds[2][1], 5)
    b_vals  = np.linspace(current_bounds[3][0], current_bounds[3][1], 5)
    
    local_best_score = -1
    local_best_params = None
    
    for xc in xc_vals:
        for yc in yc_vals:
            for a in a_vals:
                for b in b_vals:
                    score = engine.evaluate_compatibility((xc, yc, a, b))
                    if score > local_best_score:
                        local_best_score = score
                        local_best_params = (xc, yc, a, b)
                        
    if local_best_score > best_score:
        best_score = local_best_score
        best_hypothesis = local_best_params
        
    print(f"Iteration {it+1}: Best Score = {local_best_score:.2f} | Params (xc,yc,a,b) = {np.round(local_best_params, 2)}")
    
    # Refine bounds around the best parameters using Cantor middle-thirds scale shrinkage
    span_factor = 0.5 / (it + 1)
    current_bounds = [
        (max(0, local_best_params[0] - 20*span_factor), min(100, local_best_params[0] + 20*span_factor)),
        (max(0, local_best_params[1] - 20*span_factor), min(100, local_best_params[1] + 20*span_factor)),
        (max(5, local_best_params[2] - 15*span_factor), min(60,  local_best_params[2] + 15*span_factor)),
        (max(3, local_best_params[3] - 10*span_factor), min(40,  local_best_params[3] + 10*span_factor))
    ]

print("\nOptimal Fitted Ellipse Parameters:", np.round(best_hypothesis, 3))

## 4. Visualization of Results
Plotting the final extracted man-made ellipse against the original scene point cloud, proving successful noise rejection of natural/organic clutter.

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(scene_points[:, 0], scene_points[:, 1], c='lightgray', s=15, label='Scene Points (Clutter)')
plt.scatter(true_ellipse_pts[:, 0], true_ellipse_pts[:, 1], c='dodgerblue', alpha=0.6, label='True Man-Made Target')

# Draw the HRL fitted ellipse
if best_hypothesis:
    xc, yc, a, b = best_hypothesis
    angles = np.linspace(0, 2 * np.pi, 100)
    fx = xc + a * np.cos(angles)
    fy = yc + b * np.sin(angles)
    plt.plot(fx, fy, 'r-', linewidth=3, label='HRL Cantor-Fitted Ellipse')
    plt.scatter([xc], [yc], c='red', marker='x', s=100, label='Estimated Center')

plt.title('Cantor-HRL Ellipse Extraction & Noise Rejection')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.show()